In [ ]:
# Cell 1: Setup for Umbaugh 48h/72h dataset

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from importlib.metadata import version

# Scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 6))

# Paths
from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_raw = os.path.join(base_path, "Data/raw")
data_processed = os.path.join(base_path, "Data/processed")
results_path = os.path.join(base_path, "results")

umbaugh_48h72h_path = os.path.join(data_raw, "umbaugh_48h72h_GSE255834")

print("✅ Setup complete for Umbaugh 48h/72h dataset!")
print(f"Scanpy version: {version('scanpy')}")

# List files
print("\n📁 Files in directory:")
if os.path.exists(umbaugh_48h72h_path):
    files = [f for f in os.listdir(umbaugh_48h72h_path) if not f.startswith('.')]
    for f in sorted(files):
        file_path = os.path.join(umbaugh_48h72h_path, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path) / 1e6
            print(f"   {f} ({size:.1f} MB)")
else:
    print(f"   ⚠️  Directory not found: {umbaugh_48h72h_path}")

In [ ]:
# Cell 2: Load all hepatocyte samples

# Define hepatocyte files only
hep_files = [
    'GSM8081445_Ctl48_hep_raw_feature_bc_matrix.h5',
    'GSM8081447_Ctl72_hep_raw_feature_bc_matrix.h5',
    'GSM8081449_A48_hep_raw_feature_bc_matrix.h5',
    'GSM8081451_A72_hep_raw_feature_bc_matrix.h5'
]

# Sample metadata mapping
sample_info = {
    'GSM8081445_Ctl48_hep_raw_feature_bc_matrix.h5': {'treatment': 'Control', 'timepoint': '48h', 'hours': 48},
    'GSM8081447_Ctl72_hep_raw_feature_bc_matrix.h5': {'treatment': 'Control', 'timepoint': '72h', 'hours': 72},
    'GSM8081449_A48_hep_raw_feature_bc_matrix.h5': {'treatment': 'APAP', 'timepoint': '48h', 'hours': 48},
    'GSM8081451_A72_hep_raw_feature_bc_matrix.h5': {'treatment': 'APAP', 'timepoint': '72h', 'hours': 72}
}

adatas = []

print("🔄 Loading hepatocyte samples...\n")

for i, h5_file in enumerate(hep_files, 1):
    file_path = os.path.join(umbaugh_48h72h_path, h5_file)
    sample_id = h5_file.replace('_raw_feature_bc_matrix.h5', '')
    
    print(f"[{i}/4] Loading {sample_id}...")
    
    # Load H5
    adata = sc.read_10x_h5(file_path, gex_only=True)
    
    # Make gene names unique
    adata.var_names_make_unique()
    
    print(f"      Raw: {adata.n_obs:>8,} barcodes")
    
    # Fast filter: >200 genes (like Umbaugh 2h)
    genes_per_cell = (adata.X > 0).sum(axis=1).A1
    cell_filter = genes_per_cell > 200
    adata = adata[cell_filter, :].copy()
    
    print(f"      Filtered: {adata.n_obs:>6,} cells (>200 genes)")
    
    # Add unique cell IDs
    adata.obs_names = [f"{barcode}_{sample_id}" for barcode in adata.obs_names]
    
    # Add metadata
    adata.obs['sample'] = sample_id
    adata.obs['treatment'] = sample_info[h5_file]['treatment']
    adata.obs['timepoint'] = sample_info[h5_file]['timepoint']
    adata.obs['timepoint_hours'] = sample_info[h5_file]['hours']
    
    adatas.append(adata)
    print(f"      ✓ Done\n")

print("="*80)

# Concatenate
print("🔗 Concatenating all samples...")
adata_umbaugh_48h72h = sc.concat(adatas, join='outer', index_unique=None)

print("\n✅ Dataset loaded!")
print(f"📊 Total cells: {adata_umbaugh_48h72h.n_obs:,}")
print(f"📊 Total genes: {adata_umbaugh_48h72h.n_vars:,}")

print(f"\n📊 Cells per sample:")
for sample in sorted(adata_umbaugh_48h72h.obs['sample'].unique()):
    count = (adata_umbaugh_48h72h.obs['sample'] == sample).sum()
    print(f"   {sample}: {count:>6,} cells")

print(f"\n📊 Treatment distribution:")
print(adata_umbaugh_48h72h.obs['treatment'].value_counts())

print(f"\n📊 Timepoint distribution:")
print(adata_umbaugh_48h72h.obs['timepoint'].value_counts())

In [ ]:
# Cell 3: Calculate QC metrics

# Identify mitochondrial genes
adata_umbaugh_48h72h.var['mt'] = adata_umbaugh_48h72h.var_names.str.startswith('mt-')
n_mt = adata_umbaugh_48h72h.var['mt'].sum()
print(f"🔍 Found {n_mt} mitochondrial genes")

# Calculate QC
sc.pp.calculate_qc_metrics(
    adata_umbaugh_48h72h,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\n📊 QC METRICS:")
print("="*80)
print(f"📈 Median genes/cell: {adata_umbaugh_48h72h.obs['n_genes_by_counts'].median():.0f}")
print(f"📈 Median UMI/cell: {adata_umbaugh_48h72h.obs['total_counts'].median():.0f}")
print(f"📈 Median mito %: {adata_umbaugh_48h72h.obs['pct_counts_mt'].median():.2f}%")

print("\n📊 BY TREATMENT:")
for treatment in ['Control', 'APAP']:
    subset = adata_umbaugh_48h72h.obs[adata_umbaugh_48h72h.obs['treatment'] == treatment]
    print(f"\n{treatment}: {len(subset):,} cells")
    print(f"  Median genes: {subset['n_genes_by_counts'].median():.0f}")
    print(f"  Median UMI: {subset['total_counts'].median():.0f}")
    print(f"  Median mito%: {subset['pct_counts_mt'].median():.2f}%")

print("\n✅ QC calculated!")

In [ ]:
# Cell 4: Check quality and apply stricter filter if needed

median_genes = adata_umbaugh_48h72h.obs['n_genes_by_counts'].median()

print(f"📊 Current median genes/cell: {median_genes:.0f}")

if median_genes < 1000:
    print(f"\n⚠️  Low quality detected - applying >1000 genes filter...")
    
    adata_filtered = adata_umbaugh_48h72h[
        adata_umbaugh_48h72h.obs['n_genes_by_counts'] > 1000, :
    ].copy()
    
    print(f"\n   Before: {adata_umbaugh_48h72h.n_obs:,} cells")
    print(f"   After: {adata_filtered.n_obs:,} cells")
    print(f"   Removed: {adata_umbaugh_48h72h.n_obs - adata_filtered.n_obs:,} cells")
    
    adata_umbaugh_48h72h = adata_filtered.copy()
    
    print(f"\n   New median genes: {adata_umbaugh_48h72h.obs['n_genes_by_counts'].median():.0f}")
    print(f"   New median UMI: {adata_umbaugh_48h72h.obs['total_counts'].median():.0f}")
else:
    print(f"✅ Good quality - no additional filtering needed")

print("\n✅ Quality check complete!")

In [ ]:
# Cell 5: Save dataset

output_file = os.path.join(data_processed, "umbaugh_48h72h_hepatocytes.h5ad")
adata_umbaugh_48h72h.write(output_file)

print(f"✅ Dataset saved: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

print(f"\n📊 FINAL DATASET:")
print("="*80)
print(f"   Cells: {adata_umbaugh_48h72h.n_obs:,}")
print(f"   Genes: {adata_umbaugh_48h72h.n_vars:,}")
print(f"   Treatments: {adata_umbaugh_48h72h.obs['treatment'].unique().tolist()}")
print(f"   Timepoints: {sorted(adata_umbaugh_48h72h.obs['timepoint'].unique())}")
print(f"\n   Median genes/cell: {adata_umbaugh_48h72h.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_umbaugh_48h72h.obs['total_counts'].median():.0f}")
print("="*80)

In [ ]:
# Cell 5: Save dataset

output_file = os.path.join(data_processed, "umbaugh_48h72h_hepatocytes.h5ad")
adata_umbaugh_48h72h.write(output_file)

print(f"✅ Dataset saved: {output_file}")
print(f"   File size: {os.path.getsize(output_file) / 1e6:.1f} MB")

print(f"\n📊 FINAL DATASET:")
print("="*80)
print(f"   Cells: {adata_umbaugh_48h72h.n_obs:,}")
print(f"   Genes: {adata_umbaugh_48h72h.n_vars:,}")
print(f"   Treatments: {adata_umbaugh_48h72h.obs['treatment'].unique().tolist()}")
print(f"   Timepoints: {sorted(adata_umbaugh_48h72h.obs['timepoint'].unique())}")
print(f"\n   Median genes/cell: {adata_umbaugh_48h72h.obs['n_genes_by_counts'].median():.0f}")
print(f"   Median UMI/cell: {adata_umbaugh_48h72h.obs['total_counts'].median():.0f}")
print("="*80)

In [ ]:
# Cell 6: Create summary table

summary_data = []

for treatment in ['Control', 'APAP']:
    for tp in ['48h', '72h']:
        subset = adata_umbaugh_48h72h.obs[
            (adata_umbaugh_48h72h.obs['treatment'] == treatment) & 
            (adata_umbaugh_48h72h.obs['timepoint'] == tp)
        ]
        
        if len(subset) > 0:
            summary_data.append({
                'Dataset': 'Umbaugh_48h72h',
                'Treatment': treatment,
                'Timepoint': tp,
                'Cells': len(subset),
                'Median_Genes': int(subset['n_genes_by_counts'].median()),
                'Median_UMI': int(subset['total_counts'].median()),
                'Median_Mito_Pct': f"{subset['pct_counts_mt'].median():.2f}"
            })

summary_df = pd.DataFrame(summary_data)

print("📋 UMBAUGH 48H/72H SUMMARY:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

summary_file = os.path.join(data_processed, "umbaugh_48h72h_summary.csv")
summary_df.to_csv(summary_file, index=False)
print(f"\n✅ Summary saved: {summary_file}")

In [ ]:
# Cell 7: Visualize Umbaugh 48h/72h data

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Cells per treatment
treatment_counts = adata_umbaugh_48h72h.obs['treatment'].value_counts()
axes[0, 0].bar(treatment_counts.index, treatment_counts.values, color=['#2ecc71', '#e74c3c'], width=0.6)
axes[0, 0].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[0, 0].set_title('Cells per Treatment', fontweight='bold', fontsize=12)
for i, (idx, val) in enumerate(treatment_counts.items()):
    axes[0, 0].text(i, val + 50, f'{val:,}', ha='center', fontweight='bold')

# Plot 2: Cells per condition
condition_counts = adata_umbaugh_48h72h.obs.groupby(['treatment', 'timepoint']).size().unstack(fill_value=0)
condition_counts.T.plot(kind='bar', ax=axes[0, 1], color=['#2ecc71', '#e74c3c'], width=0.7)
axes[0, 1].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[0, 1].set_title('Cells per Condition', fontweight='bold', fontsize=12)
axes[0, 1].legend(['Control', 'APAP'], loc='upper right')
axes[0, 1].set_xlabel('')
axes[0, 1].tick_params(axis='x', rotation=0)

# Plot 3: Genes per cell distribution
axes[0, 2].hist(adata_umbaugh_48h72h.obs['n_genes_by_counts'], bins=100, color='#9b59b6', alpha=0.7)
axes[0, 2].axvline(adata_umbaugh_48h72h.obs['n_genes_by_counts'].median(), 
                   color='red', linestyle='--', linewidth=2, 
                   label=f"Median: {adata_umbaugh_48h72h.obs['n_genes_by_counts'].median():.0f}")
axes[0, 2].set_xlabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[0, 2].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[0, 2].set_title('Genes per Cell', fontweight='bold', fontsize=12)
axes[0, 2].legend()

# Plot 4: UMI per cell distribution
axes[1, 0].hist(adata_umbaugh_48h72h.obs['total_counts'], bins=100, color='#3498db', alpha=0.7)
axes[1, 0].axvline(adata_umbaugh_48h72h.obs['total_counts'].median(), 
                   color='red', linestyle='--', linewidth=2,
                   label=f"Median: {adata_umbaugh_48h72h.obs['total_counts'].median():.0f}")
axes[1, 0].set_xlabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 0].set_ylabel('Cells', fontweight='bold', fontsize=11)
axes[1, 0].set_title('UMI per Cell', fontweight='bold', fontsize=12)
axes[1, 0].legend()

# Plot 5: Genes vs UMI by treatment
control_data = adata_umbaugh_48h72h.obs[adata_umbaugh_48h72h.obs['treatment'] == 'Control']
apap_data = adata_umbaugh_48h72h.obs[adata_umbaugh_48h72h.obs['treatment'] == 'APAP']
axes[1, 1].scatter(control_data['total_counts'], control_data['n_genes_by_counts'], 
                   s=3, alpha=0.4, c='#2ecc71', label='Control')
axes[1, 1].scatter(apap_data['total_counts'], apap_data['n_genes_by_counts'], 
                   s=3, alpha=0.4, c='#e74c3c', label='APAP')
axes[1, 1].set_xlabel('UMI per Cell', fontweight='bold', fontsize=11)
axes[1, 1].set_ylabel('Genes per Cell', fontweight='bold', fontsize=11)
axes[1, 1].set_title('Genes vs UMI', fontweight='bold', fontsize=12)
axes[1, 1].legend()

# Plot 6: Mito % by treatment
axes[1, 2].violinplot([control_data['pct_counts_mt'], apap_data['pct_counts_mt']], 
                       positions=[0, 1], showmeans=True, showmedians=True, widths=0.7)
axes[1, 2].set_xticks([0, 1])
axes[1, 2].set_xticklabels(['Control', 'APAP'])
axes[1, 2].set_ylabel('Mitochondrial %', fontweight='bold', fontsize=11)
axes[1, 2].set_title('Mitochondrial % by Treatment', fontweight='bold', fontsize=12)

plt.tight_layout()

# Save
fig_path = os.path.join(results_path, 'figures/umbaugh_48h72h_qc.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: {fig_path}")